# Practice 4. Finding the right clause: diagnose and fix

**ML in Industry 2026 · Topic 4: Retrieval-augmented generation**

## Goal

The engineer asks again: *"After a bearing change, pump P-101 vibrates at 4.8 mm/s. Is this normal?"*
The answer is in regulation R-12, but the plant archive now has 16 documents: two revisions of R-12 and an
amendment, a near-copy of R-12 for another pump station, a maintenance regulation, a CMMS procedure, datasheets,
an English manual of an imported pump, a valve list with 110 rows and some unrelated instructions.

Search mechanisms (embeddings, hybrid search, rerankers) are taught in the Advanced NLP course. Here search is a
ready module, and we do the engineering around it:

1. a **retrieval set** built from the structure of the documents, with known evidence for every question;
2. measure the baseline and **attribute every lost answer** to a link: reading / search / context assembly / model;
3. **fix one link at a time** with plant knowledge, each fix is a row of an **ablation table** with a paired
   comparison (practice 2);
4. mechanisms from Advanced NLP are added as **rows of the same table**;
5. the answer follows a **contract**; code checks the citations; "not in the documents" and "conflict" are answers.

```
question ─► fields (tags, date) ─► registry: station, class ─► filter: scope, date ─► BM25 ─► evidence pack
                                                                                               │
                         audit log ◄── 3 citation checks ◄── answer contract (course Qwen) ◄───┘
```

## Steps and what you should see

| Step | What we do | Expected result |
|------|------------|-----------------|
| 0 | The final table | the ablation table S0-S6 and the mechanism rows |
| 1 | Setup | the course model answers; otherwise replay |
| 2 | Archive v2 | 16 documents, 440 chunks, new fields, `index_version` |
| 3 | Retrieval set | 138 questions in 12 slices, gold chunks computed by code |
| 4 | Baseline S0 | about half of the answers right; most losses in search |
| 5 | Ladder S1-S5 | tags, table rows, applicability, dates, evidence pack: one row each |
| 6 | Mechanisms | a reranker and query translation as rows of the same table |
| 7 | Answer and checks | P-101 answered with checked citations; a neighbour clause caught; conflict; abstention |
| 8 | Audit and regression | an audit record with the index version; the regression gate S6 vs S5 |
| 9 | Reproduce | the mandatory path, success criteria, optional tasks |

## Live or replay

Search, filters and evidence packs run **live** (milliseconds, no model). Model answers for all 138 questions
in every configuration come from the **replay** of the lecturer's run: identical requests get identical saved
answers. Single questions in steps 5 and 7 go to the course model live if it answers. Dense vectors and
reranker answers are replayed too (when present).

## Common problems

| Problem | What to do |
|---------|-----------|
| `Mode: REPLAY` but you have the course token | check `COURSE_LLM_BASE_URL` and `COURSE_LLM_API_KEY` in `.env`; run `uv run python -m plant_assistant.smoke_test course` |
| `ReplayMiss` | you changed a prompt, the chunking, a filter or the questions: replay works only for unchanged requests. Run live, or undo the change |
| your table differs from `reference/ablation.csv` in retrieval columns | the store differs: compare `index_version` with `reference/store/manifest.json` |
| the answer is empty, `finish_reason = length` | thinking mode is on: keep `course` (thinking off) for the answer contract |

## 0. Where we are going

The final table of this practice, measured by the lecturer. At the end you will know where every number comes from.
Every row adds one change to the row in `vs` and is compared with it **in pairs, on the same questions**.

In [1]:
import warnings

import pandas as pd
from IPython.display import Markdown, display

from plant_assistant import REFERENCE_DIR

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 250)
reference = pd.read_csv(REFERENCE_DIR / "ablation.csv").fillna("")
reference[["config", "vs", "recall@5", "evidence recall", "end-to-end", "correct + verified", "e2e verdict"]]

,config,vs,recall@5,evidence recall,end-to-end,correct + verified,e2e verdict
0,S0,,60/118 = 51% [42%–60%],53/118 = 45% [36%–54%],75/138 = 54% [46%–62%],49/138 = 36% [28%–44%],
1,S1,S0,64/118 = 54% [45%–63%],57/118 = 48% [39%–57%],80/138 = 58% [50%–66%],53/138 = 38% [31%–47%],no detectable difference (p = 0.062)
2,S2,S1,65/118 = 55% [46%–64%],55/118 = 47% [38%–56%],77/138 = 56% [47%–64%],52/138 = 38% [30%–46%],no detectable difference (p = 0.607)
3,S3,S2,78/118 = 66% [57%–74%],68/118 = 58% [49%–66%],97/138 = 70% [62%–77%],54/138 = 39% [31%–47%],B is better (p = 0.000)
4,S4,S3,86/118 = 73% [64%–80%],79/118 = 67% [58%–75%],104/138 = 75% [68%–82%],104/138 = 75% [68%–82%],B is better (p = 0.016)
5,S5,S4,86/118 = 73% [64%–80%],93/118 = 79% [71%–85%],117/138 = 85% [78%–90%],117/138 = 85% [78%–90%],B is better (p = 0.000)
6,S6,S5,86/118 = 73% [64%–80%],93/118 = 79% [71%–85%],115/138 = 83% [76%–89%],114/138 = 83% [75%–88%],no detectable difference (p = 0.500)
7,M1t,S0,8/18 = 44% [25%–66%],8/18 = 44% [25%–66%],12/20 = 60% [39%–78%],4/20 = 20% [8%–42%],B is better (p = 0.039)
8,M3@S0,S0,90/118 = 76% [68%–83%],87/118 = 74% [65%–81%],101/138 = 73% [65%–80%],56/138 = 41% [33%–49%],B is better (p = 0.000)
9,M3@S6,S6,100/118 = 85% [77%–90%],107/118 = 91% [84%–95%],124/138 = 90% [84%–94%],123/138 = 89% [83%–93%],B is better (p = 0.022)


## 1. Setup

Profiles are in `models.toml`, secrets in `.env` (as in practices 1-3). The answers use the course model
Qwen 3.8 27B with thinking off. Embedding profiles (`embed-*`) are optional: on 27.10.2026 the course server has
no embedding or reranking model, so dense vectors come from `data/replay/vectors/` when the lecturer adds them.

In [2]:
import os

from plant_assistant import REPLAY_DIR, RUNS_DIR, STORE_DIR
from plant_assistant.config import get_profile, list_profiles, load_env
from plant_assistant.llm import CallLogger, LLMClient, ReplayClient

load_env()
PROFILE = os.environ.get("PRACTICE_PROFILE", "course")
pd.DataFrame(list_profiles().items(), columns=["profile", "description"])

,profile,description
0,course,"Course model Qwen 3.8 27B via LiteLLM + vLLM, thinking off"
1,course-thinking,"Course model Qwen 3.8 27B, thinking on (spends tokens before the JSON: raise max_tokens)"
2,local,"Small open-weight model on your laptop via Ollama (Qwen3.5 4B, Q4)"
3,embed-bge-m3,Multilingual embeddings bge-m3 (567M) via Ollama: ollama pull bge-m3
4,embed-qwen3,Multilingual embeddings Qwen3-Embedding 0.6B via Ollama: ollama pull qwen3-embedding:0.6b
5,embed-course,"Embeddings on the course server, if the lab deploys a model (variant A of the design)"


In [3]:
def model_answers(profile_name: str) -> bool:
    # One call without network retries: if the model is not there, we want to know now.
    try:
        client = LLMClient(get_profile(profile_name), max_network_retries=0)
    except ValueError as exc:  # profile not configured (no URL or key)
        print(f"{profile_name}: {exc}")
        return False
    r = client.chat([{"role": "user", "content": "Reply with one word: OK"}])
    print(f"{profile_name}: {r.content.strip()[:20]!r} ({r.latency_s:.1f}s)" if r.ok else r.error[:200])
    return r.ok


LIVE = os.environ.get("PRACTICE_MODE") != "replay" and model_answers(PROFILE)
replay = {task: ReplayClient(REPLAY_DIR / f"{task}_course.jsonl") for task in ("answers", "rerank", "translate")}
live = LLMClient(get_profile(PROFILE), logger=CallLogger(RUNS_DIR / f"my-{PROFILE}" / "calls.jsonl")) if LIVE \
    else replay["answers"]
print("Mode:", f"LIVE single questions on {PROFILE}, the full set from replay" if LIVE else
      "REPLAY of the lecturer's saved answers (course model)")
vectors = sorted((REPLAY_DIR / "vectors").glob("*.npz"))
print("Embedding vectors:", [v.stem for v in vectors] or "none yet: dense rows (M1, M2) are not measured")

Mode: REPLAY of the lecturer's saved answers (course model)
Embedding vectors: none yet: dense rows (M1, M2) are not measured


## 2. The plant archive, version 2

The store of practice 3 (R-12 rev. 2 and rev. 3, two datasheets) plus 12 new **synthetic** documents. They are
born-digital: reading was topic 3 and must not get in the way. This is a teaching scale (tens of pages, hundreds
of chunks), not "thousands of pages": the process is the same, the numbers are illustrative.

`build_corpus` reads the new PDFs, takes dates and scope from the document registry (`data/documents.csv`, as from
a document management system), adds the new fields and writes `store/`.

In [4]:
from plant_assistant.corpus import build_corpus, load_registry
from plant_assistant.store import load_store

summary = build_corpus(STORE_DIR)
print(summary)
_, chunks = load_store(STORE_DIR)
by_id = {c.chunk_id: c for c in chunks}
docs = pd.DataFrame(load_registry())
docs[["doc_id", "revision", "amendment", "kind", "language", "effective_from", "effective_to", "scope_stations",
      "scope_equipment_types", "scope_tags", "pages", "source"]]

{'documents': 16, 'pages': 47, 'chunks': 440, 'chunks_without_rows': 330, 'table_rows': 110, 'index_version': 'corpus-cc3cd299df.parser-a53367.emb-none', 'seconds': 5.4}


,doc_id,revision,amendment,kind,language,effective_from,effective_to,scope_stations,scope_equipment_types,scope_tags,pages,source
0,DS-P-205,NaN,NaN,datasheet,zh,2023-01-01,None,[2],[pump],[P-205],1,practice 3
1,PS-101,NaN,NaN,datasheet,ru,2019-01-01,None,[2],[pump],[P-101],1,practice 3
2,R-12,2.0,NaN,regulation,ru,2022-06-01,2025-03-14,[2],"[pump, compressor, fan]",[],6,practice 3
3,R-12,3.0,NaN,regulation,ru,2025-03-14,None,[2],"[pump, compressor, fan]",[],6,practice 3
4,FS-07,2.0,NaN,instruction,ru,2024-01-15,None,[2],[all],[],3,practice 4
5,LAB-03,1.0,NaN,procedure,ru,2022-03-01,None,[lab],[all],[],2,practice 4
6,OM-P-205,2.0,NaN,manual,en,2023-05-15,None,[2],[pump],[P-205],3,practice 4
7,PS-102,NaN,NaN,datasheet,ru,2018-01-01,None,[2],[pump],[P-102],1,practice 4
8,PS-310,NaN,NaN,datasheet,ru,2021-01-01,None,[3],[pump],[P-310],1,practice 4
9,PS-311,NaN,NaN,datasheet,ru,2021-01-01,None,[3],[pump],[P-311],1,practice 4


**What is new, and why.** An **amendment** to R-12 rev. 3 changes one clause (4.4.1: 720 h → 500 h from
01.09.2026). **R-14** is a near-copy of R-12 for pump station No. 3, with the same title and clause numbers and a
stricter Table 3. The **manual of P-205** is in English and gives milder limits than R-12: a conflict.
The **valve list** has a table of 110 rows over four pages. R-20 defines what "priority high" in clause 6.2 of
R-12 means.

> **Chat:** "R-12 and R-14 have the same title. Name one field that tells them apart."

In [5]:
from plant_assistant.display import where

for cid in ("R-12-r3-table1", "R-14-r1-table1"):
    c = by_id[cid]
    print(f"{where(c)}  scope_stations={c.scope_stations}  valid {c.effective_from} → {c.effective_to or 'now'}")
    print("   ", c.text.splitlines()[0], "|", [line for line in c.text.splitlines() if "|II|" in line][0])

R-12 rev. 3 Table 3  scope_stations=['2']  valid 2025-03-14 → now
    Таблица 3 — Границы виброскорости по классам машин (СКЗ, мм/с) | |Насосы|II|4,5|7,1|
R-14 rev. 1 Table 3  scope_stations=['3']  valid 2024-03-01 → now
    Таблица 3 — Границы виброскорости по классам машин (СКЗ, мм/с) | |Насосы|II|3,5|5,6|


New fields of a chunk (the practice 3 schema is kept, the fields are added):
`effective_from` / `effective_to` (validity), `amended_by`, `scope_stations` / `scope_equipment_types` /
`scope_tags`, `references` / `referenced_by` (links in both directions), `table_no`, `row_key`, `doc_language`.

In [6]:
fields = ["chunk_id", "clause", "table_no", "row_key", "effective_from", "effective_to", "amended_by", "references",
          "referenced_by"]
pd.DataFrame([{f: getattr(by_id[cid], f) for f in fields} for cid in
              ("R-12-r3-4.4.1", "R-12-r3-a1-4.4.1", "R-12-r3-4.3.3", "R-12-r3-6.2", "R-20-r2-3.3",
               "VL-2-r4-tА.2-V-12")])

,chunk_id,clause,table_no,row_key,effective_from,effective_to,amended_by,references,referenced_by
0,R-12-r3-4.4.1,4.4.1,None,None,2025-03-14,2026-09-01,R-12-r3-a1-4.4.1,[],[R-12-r3-a1-4.4.1]
1,R-12-r3-a1-4.4.1,4.4.1,None,None,2026-09-01,None,None,[R-12-r3-4.4.1],[]
2,R-12-r3-4.3.3,4.3.3,None,None,2025-03-14,None,None,[R-12-r3-table1],[R-12-r3-6.2]
3,R-12-r3-6.2,6.2,None,None,2025-03-14,None,None,"[R-12-r3-4.3.3, R-12-r3-4.3.4]",[R-20-r2-3.3]
4,R-20-r2-3.3,3.3,None,None,2023-01-10,None,None,"[R-12-r2-6.2, R-12-r3-6.2, R-14-r1-6.2]",[]
5,VL-2-r4-tА.2-V-12,А,А.2,V-12,2025-11-20,None,None,[],[]


The index has a version: a fingerprint of the chunks, the parser and the embedding model. It goes into every
answer log (step 8). If any of the three changes, the index is rebuilt and the retrieval set runs again (topic 6).

In [7]:
print("index_version:", summary["index_version"])
pd.crosstab(pd.Series([c.doc_id for c in chunks], name="document"), pd.Series([c.chunk_type for c in chunks],
                                                                              name="chunk type"), margins=True)

index_version: corpus-cc3cd299df.parser-a53367.emb-none


chunk type,clause,datasheet,note,section,table,table_row,All
document,,,,,,,
DS-P-205,0,1,0,0,0,0,1
FS-07,16,0,0,5,1,0,22
LAB-03,12,0,0,4,0,0,16
OM-P-205,21,0,0,7,3,0,31
PS-101,0,1,0,0,0,0,1
PS-102,0,1,0,0,0,0,1
PS-310,0,1,0,0,0,0,1
PS-311,0,1,0,0,0,0,1
R-12,86,0,2,28,6,0,122


## 3. The retrieval set

138 questions in the format of the practice 2 harness. The 50 questions of practice 3 (known page) are the
`base` slice; 20 of them also go to search **in English as they are** (the `language` slice); the rest are new.
Every question has a Russian wording (it goes to search and to the prompt: the engineer of a Russian plant asks in
Russian) and an English translation for us. Evidence is written as **locators**; the gold chunk ids are computed
by code. Slices are reported as **counts**: 3 of 4 is not "75%".

In [8]:
from plant_assistant.cases import counts_by_slice, load_cases

cases = load_cases()
case_by_id = {c.case_id: c for c in cases}
pd.DataFrame([counts_by_slice(cases)], index=["questions"])

,base,language,applicability,point_in_time,incident,what_changed,multi_evidence,cross_doc,conflict,table_row,tag_variant,not_in_documents
questions,50,20,12,10,3,4,8,3,4,8,6,10


In [9]:
from plant_assistant.display import gold_frame

P101 = case_by_id["m01"]
print("Question (asked in Russian):", P101.question_ru)
print("Translation:                ", P101.question_en)
print("Expected:", P101.expected, "| asked on", P101.asked_on)
gold_frame(P101)

Question (asked in Russian): После замены подшипника насос P-101 вибрирует 4,8 мм/с. Это норма? Что делать и в какой срок?
Translation:                 After a bearing change pump P-101 vibrates at 4.8 mm/s. Is this normal? What must be done, and by when?
Expected: {'values': ['4,5', '24'], 'unit': 'h'} | asked on 2026-10-27


,role,locator (any of),gold chunk ids
0,answer,R-12 r3 cl 4.3.3,R-12-r3-4.3.3
1,support,R-12 r3 t 3,R-12-r3-table1
2,action,R-12 r3 cl 6.1,R-12-r3-6.1
3,action,R-12 r3 cl 6.2,R-12-r3-6.2


Four evidence chunks for one answer: the clause with the deadline (4.3.3), the table with the limit (Table 3),
and the actions (6.1, 6.2). recall@k counts only the answer; **evidence recall** asks for all four in the context.

## 4. Baseline S0: BM25 over raw chunks, top-5, no filters

> **Predict, then run.** Where will most answers be lost: reading, search, context assembly or the model?

Search runs live; the answers of the course model for 138 questions come from replay.

In [10]:
from plant_assistant import reval
from plant_assistant.display import answer_frame, context_frame, hits_frame
from plant_assistant.pipeline import CONFIGS, LADDER, System

system = System(chunks, answer_client=replay["answers"], rerank_client=replay["rerank"],
                translate_client=replay["translate"])
records, scores = {}, {}


def run(name: str) -> pd.DataFrame:
    records[name] = system.run(cases, CONFIGS[name], progress=False)
    scores[name] = reval.score(cases, records[name], by_id)
    return scores[name]


run("S0")
reval.ablation({"S0": scores["S0"]}, {})[["config", "recall@5", "evidence recall", "MRR", "end-to-end",
                                          "correct + verified", "abstain ok", "conflict ok"]]

,config,recall@5,evidence recall,MRR,end-to-end,correct + verified,abstain ok,conflict ok
0,S0,60/118 = 51% [42%–60%],53/118 = 45% [36%–54%],0.26,75/138 = 54% [46%–62%],49/138 = 36% [28%–44%],10/10,0/4


In [11]:
display(Markdown("**Wrong answers of S0 by link**"))
pd.DataFrame([reval.losses(scores["S0"])], index=["S0"])

**Wrong answers of S0 by link**

,reading,search,assembly,model
S0,0,56,2,5


Most answers are lost **in search**: the chunk with the answer never reached the model. Nothing is lost in
reading (the documents are born-digital and the practice 3 store was read well). In practice 3 the model answered
98% of these questions when it was given the page: everything below that is lost by search and assembly here.

The same view for the P-101 question: the top-5 of BM25.

In [12]:
rec = next(r for r in records["S0"] if r["case_id"] == "m01")
hits_frame(rec, P101, by_id)

,chunk,valid,gold,text
rank,,,,
1,R-12 rev. 2 cl. 4.4.2,2022-06-01 → 2025-03-14,,"4.4.2 После ремонта, замены подшипников или центровки измерения выполняют в течение 8 часов после пуска."
2,R-12 rev. 3 cl. 4.4.2,2025-03-14 → now,,"4.4.2 После ремонта, замены подшипников или центровки измерения выполняют в течение 8 часов после пуска."
3,R-14 rev. 1 cl. 4.4.2,2024-03-01 → now,,"4.4.2 После ремонта, замены подшипников или центровки измерения выполняют в течение 8 часов после пуска."
4,R-15 rev. 1 cl. 4.3,2023-09-01 → now,,4.3 Марку смазочного материала берут из паспорта насоса. Замена марки без согласования с главным механиком не
5,R-15 rev. 1 cl. 5.1,2023-09-01 → now,,5.1 Центровку валов насоса и двигателя проверяют при каждом ТО-1 и после каждого ремонта.


No word of the question is in clause 4.3.3 ("при превышении границы осмотра ... осматривают в течение 24 часов"):
lexical search cannot find it from "вибрирует 4,8 мм/с, это норма?". Only Table 3 comes up, by "мм/с".

**Task 1 of the lecture** used five failures of exactly this run:

In [13]:
display(Markdown((REFERENCE_DIR / "s0_failures.md").read_text(encoding="utf-8")))

# Five failed answers of S0 (task 1 of lecture 4)

Configuration S0: BM25 over raw chunks, top-5, no filters, the course model answers from the top-5.
Task: for each answer write where it was lost: R = reading, S = search, C = context assembly, M = model.

## 1. m01 (multi_evidence) — lost in: **search**

- Question (asked in Russian): После замены подшипника насос P-101 вибрирует 4,8 мм/с. Это норма? Что делать и в какой срок?
- Translation: After a bearing change pump P-101 vibrates at 4.8 mm/s. Is this normal? What must be done, and by when?
- Expected: ['4,5', '24'] h
- System answer: The sources do not specify the vibration limit to determine if 4.8 mm/s is normal, but they mandate that measurements be taken within 8 hours after startup following a bearing replacement. You must perform the measurement within this 8-hour window. (value: None)
- Gold evidence: R-12 r3 cl 4.3.3; R-12 r3 t 3; R-12 r3 cl 6.1; R-12 r3 cl 6.2
- Top-5 of search: R-12 rev. 2 cl. 4.4.2; R-12 rev. 3 cl. 4.4.2; R-14 rev. 1 cl. 4.4.2; R-15 rev. 1 cl. 4.3; R-15 rev. 1 cl. 5.1
- Why this link: no chunk with the answer reached the context

## 2. g01 (tag_variant) — lost in: **search**

- Question (asked in Russian): Какая мощность двигателя у насоса Р-101?
- Translation: What is the motor power of pump Р-101 (Cyrillic Р)?
- Expected: 55 kW
- System answer: The provided sources do not contain the specific power rating of the motor for pump R-101. (value: None)
- Gold evidence: PS-101
- Top-5 of search: R-12 rev. 2 cl. 4.5.1; R-12 rev. 3 cl. 4.5.1; R-14 rev. 1 cl. 4.5.1; R-15 rev. 1 cl. 5.1; R-12 rev. 2 cl. 7.2
- Why this link: no chunk with the answer reached the context

## 3. v03 (point_in_time) — lost in: **search**

- Question (asked in Russian): С какой периодичностью сейчас нужно контролировать вибрацию насоса P-101 переносным виброметром?
- Translation: How often must the vibration of pump P-101 be checked with a portable meter now?
- Expected: 500 h
- System answer: Vibration of pump P-101 must be controlled with a portable vibrometer every 1000 operating hours, but not less than once a month. (value: 1000)
- Gold evidence: R-12 r3a1 cl 4.4.1
- Top-5 of search: R-12 rev. 2 cl. 3.7; R-12 rev. 3 cl. 3.7; R-14 rev. 1 cl. 3.7; R-12 rev. 2 cl. 4.2.1; R-12 rev. 2 cl. 4.4.1
- Why this link: no chunk with the answer reached the context

## 4. m06 (multi_evidence) — lost in: **assembly**

- Question (asked in Russian): После замены подшипника насос P-101 запустили. В течение скольких часов нужно измерить вибрацию и с какой границей сравнить результат?
- Translation: Pump P-101 was started after a bearing change. Within how many hours must the vibration be measured, and against which limit?
- Expected: ['8', '4,5'] h
- System answer: Vibration measurements must be taken within 8 hours after startup. The sources do not specify the comparison limit. (value: 8)
- Gold evidence: R-12 r3 cl 4.4.2; R-12 r3 t 3
- Top-5 of search: R-12 rev. 2 cl. 4.4.2; R-12 rev. 3 cl. 4.4.2; R-14 rev. 1 cl. 4.4.2; FS-07 rev. 2 cl. 3.3; R-12 rev. 2 cl. 6.1
- Why this link: the chunk with the answer is in the context, a required support or action chunk is not

## 5. i01 (incident) — lost in: **model**

- Question (asked in Russian): 10.02.2025 насос P-101 вибрировал 4,8 мм/с, осмотр в течение суток не провели. Было ли это нарушением регламента?
- Translation: On 10 February 2025 pump P-101 vibrated at 4.8 mm/s and nobody inspected it within 24 hours. Was this a violation of the regulation?
- Expected: ['5,0'] mm/s
- System answer: Yes, it was a violation of the regulation. The vibration of 4.8 mm/s exceeded the inspection threshold of 4.5 mm/s for a Class II pump, requiring an inspection within 24 hours, which was not performed. (value: 4,5)
- Gold evidence: R-12 r2 t 3; R-12 r2 cl 4.3.3
- Top-5 of search: R-12 rev. 2 cl. 4.3.3; R-12 rev. 3 cl. 4.3.3; R-14 rev. 1 cl. 4.3.3; R-12 rev. 2 Table 3; R-12 rev. 3 Table 3
- Why this link: every needed chunk was in the context, the answer is still wrong


## 5. The ladder: one fix per row

Every row adds one change to the previous one and is compared with it in pairs. We look at one question before and
after, then at the row of the table.

**S1: tag normalization.** "Р-101" with a Cyrillic Р, "P101", "P 102": one form in the chunks **and** in the query.

In [14]:
for name in ("S0", "S1"):
    rec = system.run_case(case_by_id["g02"], CONFIGS[name], with_answer=False)
    display(Markdown(f"**{name}** · {case_by_id['g02'].question_ru}"), hits_frame(rec, case_by_id["g02"], by_id))

**S0** · Какая подача у насоса P101?

,chunk,valid,gold,text
rank,,,,
1,R-15 rev. 1 cl. 4.1,2023-09-01 → now,,4.1 Смазку подшипников качения выполняют с периодичностью по п. 7.1 R-12 для насосов станции № 2 и по п. 7.1 R
2,R-14 rev. 1 Table А.1,2024-03-01 → now,,"Таблица А.1 — Оборудование насосной станции № 3 |Тег|Наименование|Мощность<br>,<br>кВт|Класс|Паспорт| |---|---"
3,FS-07 rev. 2 cl. 2.2,2024-01-15 → now,,2.2 Проходы к насосам и электрическим щитам шириной не менее 1 м содержат свободными.
4,R-15 rev. 1 cl. 5.1,2023-09-01 → now,,5.1 Центровку валов насоса и двигателя проверяют при каждом ТО-1 и после каждого ремонта.
5,R-12 rev. 2 Table А.1,2022-06-01 → 2025-03-14,,"Таблица А.1 — Оборудование насосной станции № 2 |Тег|Наименование|Мощность<br>,<br>кВт|Класс|Паспорт| |---|---"


**S1** · Какая подача у насоса P101?

,chunk,valid,gold,text
rank,,,,
1,R-12 rev. 2 Table А.1,2022-06-01 → 2025-03-14,,"Таблица А.1 — Оборудование насосной станции № 2 |Тег|Наименование|Мощность<br>,<br>кВт|Класс|Паспорт| |---|---"
2,R-12 rev. 3 Table А.1,2025-03-14 → now,,"Таблица А.1 — Оборудование насосной станции № 2 |Тег|Наименование|Мощность<br>,<br>кВт|Класс|Паспорт| |---|---"
3,FS-07 rev. 2 Table 1,2024-01-15 → now,,Таблица 1 — Нормы оснащения помещения насосной станции № 2 |Средство|Количество|Место| |---|---|---| |Огнетуши
4,PS-101,2019-01-01 → now,answer,# ПАСПОРТ НАСОСА # PUMP DATASHEET <table> <tr> <th>Тег / Tag</th> <td>P-101</td> </tr> <tr> <th>Изго
5,R-15 rev. 1 cl. 4.1,2023-09-01 → now,,4.1 Смазку подшипников качения выполняют с периодичностью по п. 7.1 R-12 для насосов станции № 2 и по п. 7.1 R


**S2: rows of long tables are chunks.** The valve list has 110 rows on 4 pages. A page of 33 rows is one chunk in
S0-S1; in S2 every row is a chunk with the caption, the column names and the units:

In [15]:
print(by_id["VL-2-r4-tА.2-V-12"].text)
for name in ("S1", "S2"):
    rec = system.run_case(case_by_id["r01"], CONFIGS[name], with_answer=False)
    display(Markdown(f"**{name}** · {case_by_id['r01'].question_ru}"), hits_frame(rec, case_by_id["r01"], by_id, 3))

Таблица А.2 — Перечень запорной арматуры насосной станции № 2. Строка V-12. Тег: V-12; Тип: ЗД; DN, мм: 150; PN, МПа: 1,6; Среда: нефть; Трубопровод: нагнетание P-101; Привод: ручной


**S1** · Какое допустимое давление для задвижки V-12?

,chunk,valid,gold,text
rank,,,,
1,VL-2 rev. 4 cl. 1.3,2025-11-20 → now,support,"1.3 Арматуру эксплуатируют при рабочем давлении, не превышающем номинальное давление PN, указанное в таблице А"
2,R-15 rev. 1 cl. 6.1,2023-09-01 → now,,6.1 Допустимая утечка через торцевое уплотнение — не более 10 капель в минуту.
3,R-15 rev. 1 cl. 6.2,2023-09-01 → now,,6.2 При превышении допустимой утечки начальник смены создаёт заявку с приоритетом «средний» по R-20.


**S2** · Какое допустимое давление для задвижки V-12?

,chunk,valid,gold,text
rank,,,,
1,VL-2 rev. 4 cl. 1.3,2025-11-20 → now,support,"1.3 Арматуру эксплуатируют при рабочем давлении, не превышающем номинальное давление PN, указанное в таблице А"
2,VL-2 rev. 4 Table А.2 row V-12,2025-11-20 → now,answer,"Таблица А.2 — Перечень запорной арматуры насосной станции № 2. Строка V-12. Тег: V-12; Тип: ЗД; DN, мм: 150; P"
3,R-15 rev. 1 cl. 6.1,2023-09-01 → now,,6.1 Допустимая утечка через торцевое уплотнение — не более 10 капель в минуту.


**S3: applicability through the registry.** The tag goes to the equipment registry: station, type, class. The
filter keeps only documents whose scope fits, **before** search; the class goes into the prompt. P-310 is at
station No. 3, so R-14 applies, not R-12.

In [16]:
from plant_assistant.applicability import equipment_context

system_live = System(chunks, answer_client=live)  # single questions: live if the course model answers
for name in ("S2", "S3"):
    rec = system_live.run_case(case_by_id["p01"], CONFIGS[name])
    display(Markdown(f"**{name}** · {case_by_id['p01'].question_ru} · scope: {rec['scope'] or 'none'}"),
            hits_frame(rec, case_by_id["p01"], by_id, 3))
    print("answer:", (rec["answer"] or {}).get("answer"))

**S2** · Насос P-310 вибрирует 4,0 мм/с. Нужно ли его осматривать? · scope: none

,chunk,valid,gold,text
rank,,,,
1,R-14 rev. 1 Table А.1,2024-03-01 → now,support,"Таблица А.1 — Оборудование насосной станции № 3 |Тег|Наименование|Мощность<br>,<br>кВт|Класс|Паспорт| |---|---"
2,R-12 rev. 2 Table 3,2022-06-01 → 2025-03-14,,"Таблица 3 — Границы виброскорости по классам машин (СКЗ, мм/с) |Группа оборудования|Класс<br>машины|Граница ви"
3,R-12 rev. 3 Table 3,2025-03-14 → now,,"Таблица 3 — Границы виброскорости по классам машин (СКЗ, мм/с) |Группа оборудования|Класс<br>машины|Граница ви"


answer: Yes, pump P-310 requires inspection because its vibration of 4.0 mm/s exceeds the inspection threshold of 3.5 mm/s for Class II pumps. The pump should be inspected immediately.


**S3** · Насос P-310 вибрирует 4,0 мм/с. Нужно ли его осматривать? · scope: stations=['3'], types=['pump'], tags=['P-310']

,chunk,valid,gold,text
rank,,,,
1,R-14 rev. 1 Table А.1,2024-03-01 → now,support,"Таблица А.1 — Оборудование насосной станции № 3 |Тег|Наименование|Мощность<br>,<br>кВт|Класс|Паспорт| |---|---"
2,R-14 rev. 1 Table 3,2024-03-01 → now,answer,"Таблица 3 — Границы виброскорости по классам машин (СКЗ, мм/с) |Группа оборудования|Класс<br>машины|Граница ви"
3,R-14 rev. 1 cl. 3.3,2024-03-01 → now,,"3.3 Граница осмотра: значение виброскорости, при превышении которого оборудование осматривают в установленный"


answer: Yes, pump P-310 must be inspected because its vibration of 4.0 mm/s exceeds the inspection limit of 3.5 mm/s for machine class II. The inspection must be performed within 24 hours.


**S4: a revision filter by the date of the question.** `effective_from <= date < effective_to`, not
`status = in force`. The same question on three dates: 1000 h (rev. 2), 720 h (rev. 3), 500 h (amendment No. 1).
At S4 the old wordings are gone, but the amendment is not in the top-5 for "now" (its text differs from the
question): the model says "not in the documents". The evidence pack of S5 brings it as a part of section 4.4.

In [17]:
for cid in ("v01", "v02", "v03"):
    for name in ("S3", "S4", "S5"):
        rec = system_live.run_case(case_by_id[cid], CONFIGS[name])
        a = rec["answer"] or {}
        print(f"{cid} {name}: as_of {rec['as_of']}  expected {case_by_id[cid].expected['value']} h  "
              f"answer {a.get('value')} {a.get('unit')}  top-1 {where(by_id[rec['retrieved'][0]])}")

v01 S3: as_of 2024-05-01  expected 1000 h  answer 720 hours  top-1 R-12 rev. 2 cl. 4.4.1
v01 S4: as_of 2024-05-01  expected 1000 h  answer 1000 hours  top-1 R-12 rev. 2 cl. 4.4.1
v01 S5: as_of 2024-05-01  expected 1000 h  answer 1000 hours  top-1 R-12 rev. 2 cl. 4.4.1
v02 S3: as_of 2025-06-01  expected 720 h  answer 720 hours  top-1 R-12 rev. 2 cl. 4.4.1
v02 S4: as_of 2025-06-01  expected 720 h  answer 720 hours  top-1 R-12 rev. 3 cl. 4.4.1
v02 S5: as_of 2025-06-01  expected 720 h  answer 720 hours  top-1 R-12 rev. 3 cl. 4.4.1
v03 S3: as_of 2026-10-27  expected 500 h  answer 1000 hours  top-1 R-12 rev. 2 cl. 3.7
v03 S4: as_of 2026-10-27  expected 500 h  answer None None  top-1 R-12 rev. 3 cl. 3.7
v03 S5: as_of 2026-10-27  expected 500 h  answer 500 hours  top-1 R-12 rev. 3 cl. 3.7


**Typical mistakes, measured.** A filter *after* search empties the top-k; the flag `status = in force` answers a
question about an incident in February 2025 with rev. 3.

In [18]:
from plant_assistant.filters import make_filter, post_filter, status_in_force
from plant_assistant.query import extract_rules

q = case_by_id["i01"]
fields = extract_rules(q.query, q.asked_on)
idx = system.bm25(CONFIGS["S4"])
by_date = make_filter(fields, None, use_scope=False, use_date=True)
pre = [where(by_id[h.chunk_id]) for h in idx.search(q.query, 5, by_date)]
post = [where(by_id[h.chunk_id]) for h in post_filter(idx.search(q.query, 5), by_id, by_date)]
flag = [where(by_id[h.chunk_id]) for h in idx.search(q.query, 5, status_in_force)]
print("question:", q.question_en)
pd.DataFrame({"pre-filter by date": pd.Series(pre), "post-filter by date": pd.Series(post),
              "status = in force": pd.Series(flag)})

question: On 10 February 2025 pump P-101 vibrated at 4.8 mm/s and nobody inspected it within 24 hours. Was this a violation of the regulation?


,pre-filter by date,post-filter by date,status = in force
0,R-12 rev. 2 Table 3,R-12 rev. 2 Table 3,R-12 rev. 3 Table 3
1,R-14 rev. 1 Table 3,R-14 rev. 1 Table 3,R-14 rev. 1 Table 3
2,R-12 rev. 2 cl. 4.3.3,R-12 rev. 2 cl. 4.3.3,R-12 rev. 3 cl. 4.3.3
3,R-14 rev. 1 cl. 4.3.3,NaN,R-14 rev. 1 cl. 4.3.3
4,R-12 rev. 2 cl. 6.1,NaN,R-12 rev. 3 cl. 6.1


**S5: the evidence pack.** For every hit: the chunks it refers to and the chunks that refer to it (6.2 refers to
4.3.3, not the other way round), one more step back, then the whole sections, within 2500 tokens.

In [19]:
for name in ("S4", "S5"):
    rec = system.run_case(P101, CONFIGS[name], with_answer=False)
    ctx = context_frame(rec, P101, by_id)
    print(f"{name}: {len(ctx)} chunks, {ctx.tokens.sum()} tokens; gold in the context: "
          f"{sorted(set(ctx.gold) - {''})}, evidence recall = {reval.retrieval_metrics(P101, rec)['evidence_recall']}")
ctx[ctx.gold != ""]

S4: 5 chunks, 255 tokens; gold in the context: ['support'], evidence recall = False
S5: 33 chunks, 1285 tokens; gold in the context: ['action', 'answer', 'support'], evidence recall = True


,chunk,gold,tokens
3,R-12 rev. 3 Table 3,support,115
7,R-12 rev. 3 cl. 4.3.3,answer,46
8,R-12 rev. 3 cl. 6.2,action,64
18,R-12 rev. 3 cl. 6.1,action,32


> **Poll:** "Which fix will give the biggest gain on the whole set? Tag normalization / Table rows /
> Applicability / Revision filter / Evidence pack"

The rows of the table, each against the previous one:

In [20]:
for name in ("S1", "S2", "S3", "S4", "S5"):
    run(name)
previous = {"S1": "S0", "S2": "S1", "S3": "S2", "S4": "S3", "S5": "S4", "S6": "S5", "M1t": "S0", "M3@S0": "S0",
            "M3@S6": "S6"}
cols = ["config", "vs", "recall@5", "evidence recall", "end-to-end", "e2e: fixed / broke", "e2e verdict",
        "evidence verdict"]
reval.ablation(scores, previous)[cols].fillna("")

,config,vs,recall@5,evidence recall,end-to-end,e2e: fixed / broke,e2e verdict,evidence verdict
0,S0,,60/118 = 51% [42%–60%],53/118 = 45% [36%–54%],75/138 = 54% [46%–62%],,,
1,S1,S0,64/118 = 54% [45%–63%],57/118 = 48% [39%–57%],80/138 = 58% [50%–66%],+5 / -0,no detectable difference (p = 0.062),"no detectable difference (+4/-0, p = 0.125)"
2,S2,S1,65/118 = 55% [46%–64%],55/118 = 47% [38%–56%],77/138 = 56% [47%–64%],+6 / -9,no detectable difference (p = 0.607),"no detectable difference (+3/-5, p = 0.727)"
3,S3,S2,78/118 = 66% [57%–74%],68/118 = 58% [49%–66%],97/138 = 70% [62%–77%],+21 / -1,B is better (p = 0.000),"B is better (+14/-1, p = 0.001)"
4,S4,S3,86/118 = 73% [64%–80%],79/118 = 67% [58%–75%],104/138 = 75% [68%–82%],+7 / -0,B is better (p = 0.016),"B is better (+11/-0, p = 0.001)"
5,S5,S4,86/118 = 73% [64%–80%],93/118 = 79% [71%–85%],117/138 = 85% [78%–90%],+13 / -0,B is better (p = 0.000),"B is better (+14/-0, p = 0.000)"


In [21]:
display(Markdown("**End-to-end by slice** (counts)"))
reval.by_slice(scores, "correct")

**End-to-end by slice** (counts)

,base,language,applicability,point_in_time,incident,what_changed,multi_evidence,cross_doc,conflict,table_row,tag_variant,not_in_documents
config,,,,,,,,,,,,
S0,43/50,5/20,2/12,4/10,1/3,1/4,3/8,0/3,0/4,6/8,0/6,10/10
S1,43/50,5/20,2/12,4/10,1/3,1/4,3/8,0/3,0/4,6/8,5/6,10/10
S2,38/50,5/20,4/12,4/10,0/3,1/4,3/8,0/3,0/4,8/8,4/6,10/10
S3,44/50,10/20,9/12,5/10,2/3,1/4,3/8,0/3,0/4,8/8,5/6,10/10
S4,44/50,10/20,11/12,9/10,3/3,1/4,3/8,0/3,0/4,8/8,5/6,10/10
S5,44/50,11/20,12/12,10/10,3/3,3/4,8/8,3/3,0/4,8/8,5/6,10/10


Read the table by slices, not only by the total: S2 **breaks** some base questions (110 short row chunks with
pump tags compete with the datasheets) and S3 fixes them; S4 is where `point_in_time` and `incident` move.

## 6. Mechanisms from Advanced NLP: rows of the same table

Two points of attachment: **instead of** plant knowledge (on S0) and **on top of** it (on S6). We do not explain
how they work; we look at where they help.

- **M3** an LLM reranker (listwise, course Qwen) over the top-20 of the current search;
- **M1t** translation of an English question into Russian before search (the `language` slice only);
- **M1 / M2** dense and hybrid search: need embedding vectors (see below).

> **Chat:** "Will a reranker fix the point-in-time slice? yes / no"

In [22]:
for name in ("S6", "M1t", "M3@S0", "M3@S6"):
    run(name)
table = reval.ablation(scores, previous)
table[table.config.isin(["S0", "M1t", "M3@S0", "S6", "M3@S6"])][cols].fillna("")

,config,vs,recall@5,evidence recall,end-to-end,e2e: fixed / broke,e2e verdict,evidence verdict
0,S0,,60/118 = 51% [42%–60%],53/118 = 45% [36%–54%],75/138 = 54% [46%–62%],,,
6,S6,S5,86/118 = 73% [64%–80%],93/118 = 79% [71%–85%],115/138 = 83% [76%–89%],+0 / -2,no detectable difference (p = 0.500),"no detectable difference (+0/-0, p = 1.000)"
7,M1t,S0,8/18 = 44% [25%–66%],8/18 = 44% [25%–66%],12/20 = 60% [39%–78%],+8 / -1,B is better (p = 0.039),"no detectable difference (+7/-2, p = 0.180)"
8,M3@S0,S0,90/118 = 76% [68%–83%],87/118 = 74% [65%–81%],101/138 = 73% [65%–80%],+30 / -4,B is better (p = 0.000),"B is better (+36/-2, p = 0.000)"
9,M3@S6,S6,100/118 = 85% [77%–90%],107/118 = 91% [84%–95%],124/138 = 90% [84%–94%],+11 / -2,B is better (p = 0.022),"B is better (+14/-0, p = 0.000)"


In [23]:
reval.by_slice({k: scores[k] for k in ("S0", "M3@S0", "S6", "M3@S6")}, "correct")

,base,language,applicability,point_in_time,incident,what_changed,multi_evidence,cross_doc,conflict,table_row,tag_variant,not_in_documents
config,,,,,,,,,,,,
S0,43/50,5/20,2/12,4/10,1/3,1/4,3/8,0/3,0/4,6/8,0/6,10/10
M3@S0,47/50,10/20,9/12,6/10,2/3,3/4,3/8,1/3,0/4,8/8,2/6,10/10
S6,44/50,11/20,12/12,10/10,2/3,3/4,8/8,2/3,0/4,8/8,5/6,10/10
M3@S6,49/50,13/20,11/12,10/10,3/3,3/4,8/8,3/3,0/4,8/8,6/6,10/10


In [24]:
lang = scores["M1t"].case_id
display(Markdown("**The `language` slice**: English questions as they are (S0) vs translated to Russian (M1t)"))
pd.DataFrame({name: [f"{int(scores[name].set_index('case_id').loc[lang, m].astype(bool).sum())}/{len(lang)}"
                     for m in ("recall@k", "correct")] for name in ("S0", "M1t")}, index=["recall@5", "end-to-end"])

**The `language` slice**: English questions as they are (S0) vs translated to Russian (M1t)

,S0,M1t
recall@5,3/20,8/20
end-to-end,5/20,12/20


**Dense search (M1, M2).** On 27.10.2026 the course server has no embedding model, and the lecturer's vectors are
not in `data/replay/vectors/` yet. With Ollama you can make them yourself (optional task 2):
`ollama pull bge-m3`, then `uv run python scripts/build_vectors.py --profile embed-bge-m3 --name bge-m3`.
The cell below runs M1 and M2 when vectors are there.

In [25]:
from dataclasses import replace

from plant_assistant.search import VectorStore

if vectors:
    system.vectors = VectorStore(vectors[0])
    system.vectors.check(system.index_version)
    for name in ("M1@S0", "M2@S0", "M1@S6", "M2@S6"):
        run(name)
    previous.update({"M1@S0": "S0", "M2@S0": "S0", "M1@S6": "S6", "M2@S6": "S6"})
    display(reval.ablation(scores, previous)[cols].fillna(""))
else:
    print("No vectors: M1 and M2 are TODO in the reference table too.")

No vectors: M1 and M2 are TODO in the reference table too.


## 7. The answer contract and the citation checks

S6 = S5 + the answer contract: a status (`answered`, `not_in_documents`, `conflict`, `needs_human`), the value with
its unit, claims with their sources, and citations as fields that code can check:

1. **exists**: a chunk with this document, revision, clause / table / row is in the index;
2. **in force**: it is valid on the date of the question;
3. **supports**: the number and the unit of the answer are in its text (or in a table it refers to).

The P-101 question, live on the course model if it answers:

In [26]:
rec = system_live.run_case(P101, CONFIGS["S6"])
answer, checks = answer_frame(rec)
print("from replay" if rec["from_replay"] else f"live, {rec['latency_s']} s", "| verified:", rec["checks"]["verified"])
display(answer)
checks

from replay | verified: True


,value
field,
status,answered
answer,The vibration level of 4.8 mm/s exceeds the inspection limit (4.5 mm/s) for a Class II pump but is below the stop li...
value,"4,5"
unit,мм/с


,source,chunk_id,exists,in_force,supports
0,S4,R-12-r3-table1,True,True,True
1,S8,R-12-r3-4.3.3,True,True,True
2,S19,R-12-r3-6.1,True,True,False
3,S9,R-12-r3-6.2,True,True,False
4,S20,R-12-r3-6.3,True,True,False


> **Chat:** "The answer cites R-12 rev. 2. Which check fails?"

Task 3 of the lecture: *"Inspect P-101 within 24 hours (R-12, rev. 3, clause 4.3.4, page 3)"*. The clause exists
and is in force; its text is about the **stop** limit, "24 hours" is in the neighbouring clause 4.3.3.

In [27]:
from plant_assistant.answer import Answer, Citation
from plant_assistant.verify import check_answer

claim = Answer(status="answered", answer="Inspect P-101 within 24 hours.", value="24", unit="h",
               citations=[Citation(source="S1", doc_id="R-12", revision=3, clause="4.3.4", page=3)])
rows = []
for rev, clause in ((3, "4.3.4"), (3, "4.3.3"), (2, "4.3.3")):
    claim.citations[0].revision, claim.citations[0].clause = rev, clause
    r = check_answer(claim, [], chunks, by_id, P101.asked_on)
    rows.append({"citation": f"R-12 rev. {rev} cl. {clause}", "exists": r["exists"], "in force": r["in_force"],
                 "supports 24 h": r["supports"], "verified": r["verified"]})
pd.DataFrame(rows)

,citation,exists,in force,supports 24 h,verified
0,R-12 rev. 3 cl. 4.3.4,True,True,False,False
1,R-12 rev. 3 cl. 4.3.3,True,True,True,True
2,R-12 rev. 2 cl. 4.3.3,True,False,True,False


**Conflict.** P-205 is a class II pump at station No. 2: R-12 says inspect above 4.5 mm/s; the manufacturer's
manual says "alarm 7.1 mm/s". The system must show both, not choose. First, what S6 does with the question as asked:

In [28]:
rec = system_live.run_case(case_by_id["f01"], CONFIGS["S6"])
print("S6 status:", rec["answer"]["status"], "|", rec["answer"]["answer"])
print("manual of P-205 in the context:", any(c.startswith("OM-P-205") for c in rec["context"]))

S6 status: answered | The inspection limit (граница осмотра) for pump P-205 (machine class II) is 4.5 mm/s. Upon exceeding this limit, the equipment must be inspected within 24 hours, and measurements must be repeated every shift until the inspection is completed. The shift supervisor must create a CMMS request with 'high' priority.
manual of P-205 in the context: False


The Russian question does not find the English manual with BM25 (no shared words except numbers written
differently, and the tables of the manual do not even name the tag). A conflict that search does not bring is a
conflict the system cannot show. When both sides are in the context, the contract does its job:

In [29]:
from plant_assistant.pipeline import CONFLICT_DEMO

demo = System(chunks, answer_client=live).answer_from_context(case_by_id[CONFLICT_DEMO[0]], CONFLICT_DEMO[1],
                                                               CONFIGS["S6"])
print("status:", demo["answer"]["status"])
print(demo["answer"]["answer"])

status: conflict
The sources provide different vibration limits for pump P-205 (Class II). Source [S1] sets the inspection limit at 4.5 mm/s and the stop limit at 7.1 mm/s. Source [S2] sets the alarm limit at 7.1 mm/s (with a 7-day deadline to find the cause) and the trip limit at 11.2 mm/s. The values for the alarm/inspection threshold and the stop/trip threshold differ between the plant regulation and the equipment manual.


**Not in the documents.** V-912 is in no document (V-112 is in the registry but in no document either):

In [30]:
for cid in ("x01", "x03"):
    rec = system_live.run_case(case_by_id[cid], CONFIGS["S6"])
    print(cid, rec["answer"]["status"], "|", rec["answer"]["answer"][:160])

x01 not_in_documents | The sources do not contain the specific allowable pressure (PN) for valve V-912. Clause 1.3 of VL-2 states that the working pressure must not exceed the nominal
x03 not_in_documents | The provided sources do not contain information about the allowable pressure for valve V-112. The sources only describe the CMMS maintenance request procedure.


The whole ladder with S6:

In [31]:
full = reval.ablation(scores, previous)
full[full.config.isin([c.name for c in LADDER])][cols + ["correct + verified", "abstain ok", "conflict ok"]].fillna("")

,config,vs,recall@5,evidence recall,end-to-end,e2e: fixed / broke,e2e verdict,evidence verdict,correct + verified,abstain ok,conflict ok
0,S0,,60/118 = 51% [42%–60%],53/118 = 45% [36%–54%],75/138 = 54% [46%–62%],,,,49/138 = 36% [28%–44%],10/10,0/4
1,S1,S0,64/118 = 54% [45%–63%],57/118 = 48% [39%–57%],80/138 = 58% [50%–66%],+5 / -0,no detectable difference (p = 0.062),"no detectable difference (+4/-0, p = 0.125)",53/138 = 38% [31%–47%],10/10,0/4
2,S2,S1,65/118 = 55% [46%–64%],55/118 = 47% [38%–56%],77/138 = 56% [47%–64%],+6 / -9,no detectable difference (p = 0.607),"no detectable difference (+3/-5, p = 0.727)",52/138 = 38% [30%–46%],10/10,0/4
3,S3,S2,78/118 = 66% [57%–74%],68/118 = 58% [49%–66%],97/138 = 70% [62%–77%],+21 / -1,B is better (p = 0.000),"B is better (+14/-1, p = 0.001)",54/138 = 39% [31%–47%],10/10,0/4
4,S4,S3,86/118 = 73% [64%–80%],79/118 = 67% [58%–75%],104/138 = 75% [68%–82%],+7 / -0,B is better (p = 0.016),"B is better (+11/-0, p = 0.001)",104/138 = 75% [68%–82%],10/10,0/4
5,S5,S4,86/118 = 73% [64%–80%],93/118 = 79% [71%–85%],117/138 = 85% [78%–90%],+13 / -0,B is better (p = 0.000),"B is better (+14/-0, p = 0.000)",117/138 = 85% [78%–90%],10/10,0/4
6,S6,S5,86/118 = 73% [64%–80%],93/118 = 79% [71%–85%],115/138 = 83% [76%–89%],+0 / -2,no detectable difference (p = 0.500),"no detectable difference (+0/-0, p = 1.000)",114/138 = 83% [75%–88%],10/10,0/4


## 8. The audit log and the regression gate

Half a year later, "why did the system say 5,0?" must have an answer. The record keeps the question, its date,
the index version, the chunks with their revisions, the answer and the checks.

In [32]:
import json

from plant_assistant.audit import audit_record

rec = next(r for r in records["S6"] if r["case_id"] == "m01")
print(json.dumps(audit_record(rec, system.index_version, by_id), ensure_ascii=False, indent=1)[:1800])

{
 "ts": "2026-10-27T17:39:25+00:00",
 "case_id": "m01",
 "question": "После замены подшипника насос P-101 вибрирует 4,8 мм/с. Это норма? Что делать и в какой срок?",
 "asked_on": "2026-10-27",
 "as_of": "2026-10-27",
 "config": "S6",
 "index_version": "corpus-cc3cd299df.parser-a53367.emb-none",
 "filters": "stations=['2'], types=['pump'], tags=['P-101']",
 "retrieved": [
  {
   "chunk_id": "R-12-r3-4.4.2",
   "doc_id": "R-12",
   "revision": 3,
   "amendment": null,
   "clause": "4.4.2",
   "table_no": null,
   "row_key": null,
   "page": 4,
   "effective_from": "2025-03-14",
   "effective_to": null
  },
  {
   "chunk_id": "R-15-r1-4.3",
   "doc_id": "R-15",
   "revision": 1,
   "amendment": null,
   "clause": "4.3",
   "table_no": null,
   "row_key": null,
   "page": 3,
   "effective_from": "2023-09-01",
   "effective_to": null
  },
  {
   "chunk_id": "R-15-r1-5.1",
   "doc_id": "R-15",
   "revision": 1,
   "amendment": null,
   "clause": "5.1",
   "table_no": null,
   "row_key": nul

The retrieval set is a regression test: the gate of practice 2 (not significantly worse, at most 2 questions
broken) on S6 against S5. Topic 6 puts this into CI and runs it after every new revision or index version.

In [33]:
g = reval.gate(scores["S5"], scores["S6"])
print(f"S6 vs S5: {g['A right']} -> {g['B right']} of {g['n']}, fixed {g['B fixed']}, broke {g['B broke']} "
      f"{g['broken_cases']}, p = {g['p_value']}  ->  GATE {'PASS' if g['passed'] else 'FAIL'} {g['reasons']}")

S6 vs S5: 117 -> 115 of 138, fixed 0, broke 2 ['i01', 'k02'], p = 0.5  ->  GATE PASS []


## 9. Check yourself and reproduce

**Your table against the reference.** Retrieval columns must match exactly (BM25 and filters are deterministic);
end-to-end from replay matches too, a live run may differ by a few questions (look at the intervals).

In [34]:
mine = reval.ablation(scores, previous).set_index("config")
ref = reference.set_index("config")
common = [c for c in mine.index if c in ref.index]
pd.DataFrame({"recall@5 (yours)": mine.loc[common, "recall@5"], "recall@5 (reference)": ref.loc[common, "recall@5"],
              "end-to-end (yours)": mine.loc[common, "end-to-end"],
              "end-to-end (reference)": ref.loc[common, "end-to-end"]})

,recall@5 (yours),recall@5 (reference),end-to-end (yours),end-to-end (reference)
config,,,,
S0,60/118 = 51% [42%–60%],60/118 = 51% [42%–60%],75/138 = 54% [46%–62%],75/138 = 54% [46%–62%]
S1,64/118 = 54% [45%–63%],64/118 = 54% [45%–63%],80/138 = 58% [50%–66%],80/138 = 58% [50%–66%]
S2,65/118 = 55% [46%–64%],65/118 = 55% [46%–64%],77/138 = 56% [47%–64%],77/138 = 56% [47%–64%]
S3,78/118 = 66% [57%–74%],78/118 = 66% [57%–74%],97/138 = 70% [62%–77%],97/138 = 70% [62%–77%]
S4,86/118 = 73% [64%–80%],86/118 = 73% [64%–80%],104/138 = 75% [68%–82%],104/138 = 75% [68%–82%]
S5,86/118 = 73% [64%–80%],86/118 = 73% [64%–80%],117/138 = 85% [78%–90%],117/138 = 85% [78%–90%]
S6,86/118 = 73% [64%–80%],86/118 = 73% [64%–80%],115/138 = 83% [76%–89%],115/138 = 83% [76%–89%]
M1t,8/18 = 44% [25%–66%],8/18 = 44% [25%–66%],12/20 = 60% [39%–78%],12/20 = 60% [39%–78%]
M3@S0,90/118 = 76% [68%–83%],90/118 = 76% [68%–83%],101/138 = 73% [65%–80%],101/138 = 73% [65%–80%]


**Success criteria**

- the retrieval columns of your ablation table equal `reference/ablation.csv`;
- the P-101 question at S6: "not normal, inspect within 24 hours", citations R-12 rev. 3 cl. 4.3.3, Table 3,
  cl. 6.1-6.2, all three checks pass;
- V-912: `not_in_documents`; P-205 with both sources in the context: `conflict`.

**What you built:** a retrieval set with evidence from the structure of the documents, loss attribution by link,
tag normalization, table rows, applicability through the registry, a date filter that understands amendments,
an evidence pack with references in both directions, an answer contract with three citation checks, an audit
record with the index version, and the regression gate. In the demo project this becomes the search tool of the
agent (topic 5), the regression test of the index (topic 6) and the audit log (topics 6 and 7).

**Typical mistakes** (check before you trust a number):
- the revision filter after search: the top-k is full of the old revision, after the filter nothing is left;
- `status = in force` instead of the dates: a wrong answer to a question about an incident;
- amendment No. 1 not applied: clause 4.4.1 of rev. 3 looks valid;
- the tag normalized in the chunks but not in the question (or the other way round);
- the date filter on for a "what changed" question: rev. 2 disappears;
- "dense is better than BM25" from two averages without a paired comparison; a percentage on a slice of 4;
- a citation check that only looks for the clause: the neighbouring clause 4.3.4 passes;
- `question_en` sent to search instead of `question_ru`: BM25 finds only tags and numbers;
- a new embedding profile with the old vectors: the manifest (`index_version`) must catch it;
- the course model in thinking mode spent the token limit before the JSON of the contract.

**Optional tasks** (see the README for details):
1. Reading against search: S6 on a store read from the scan by Docling (practice 3). How many losses move to "reading"?
2. Your own embedder: `bge-m3` against `qwen3-embedding:0.6b` via Ollama, paired comparison on the `language` slice.
3. LLM extraction of the query fields (`query.extract_llm`) against the rules: where do the rules miss a date or a tag?
4. Context budget: the evidence pack at 500 / 1000 / 2000 tokens: evidence recall and end-to-end.
5. Near-duplicates without metadata: remove `scope` and count the P-310 questions answered from R-12.
6. Document hierarchy: a company standard above R-12 and the rule "the stricter wins" in code; the `conflict` slice.
7. A vector database: the index in pgvector or Qdrant (installed from PyPI or a mirror); time and result.
8. Your documents: the process on your project documents: 20 questions from the structure, S0 and one ladder row.